# Municipal 311 Assistant — Adapter Sprint (QLoRA + DPO under a 15GB ceiling)

Rewrites messy resident 311 complaints into structured dispatch tickets.

Pipeline: synthetic data -> base model eval -> QLoRA SFT -> deliberate OOM/diagnosis/fix -> DPO -> comparative eval.

**Run this end-to-end on a Colab T4 runtime.** Every number in the results tables below is written by the
code that runs it (`results.json` / `results.csv`) — nothing in this notebook is a pre-filled measurement.
If you haven't run a cell, its results are not real yet.

## 0. Setup

Model: **`Qwen/Qwen2.5-0.5B-Instruct`** (0.5B params, Apache-2.0). Chosen because it's already
instruction-tuned (so the base-model baseline is meaningful), small enough that 4-bit quantization
leaves most of the T4's ~15GB for activations/optimizer state, and fully supported by current
`transformers`/`peft`/`bitsandbytes`/`trl` without custom code.

In [ ]:
!pip install -q -U transformers peft bitsandbytes trl accelerate datasets


In [ ]:
import os, json, time, random, re, traceback, gc
from dataclasses import dataclass, field

import torch
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LEN = 256          # complaints + tickets are short; keep activation memory proportional to real data
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(OUT_DIR, "results.json")

def log_result(record: dict, path: str = RESULTS_PATH):
    """Append one measured record (never hand-typed) to the running results log."""
    records = []
    if os.path.exists(path):
        with open(path) as f:
            records = json.load(f)
    records.append(record)
    with open(path, "w") as f:
        json.dump(records, f, indent=2)
    return record

def profile_run(stage_name: str, fn):
    """Run fn(), measuring wall-clock time and peak CUDA memory actually allocated during the call."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    result = fn()
    torch.cuda.synchronize()
    elapsed_sec = round(time.time() - t0, 1)
    peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 3)
    record = log_result({"stage": stage_name, "peak_gpu_mem_gb": peak_gb, "wall_time_sec": elapsed_sec})
    print(f"[{stage_name}] peak_gpu_mem_gb={peak_gb}  wall_time_sec={elapsed_sec}")
    return result, record

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — this notebook needs a GPU runtime")


## 1. Synthetic 311 complaint + ticket data

Fully offline, rule/template based — no external API calls, so it's reproducible and free.
Each example pairs a "messy" resident complaint with the structured dispatch ticket it should produce.

Ticket schema:
```json
{"category": str, "department": str, "priority": "low|medium|high", "location": str,
 "description": str, "requested_action": str}
```

In [ ]:
CATEGORIES = {
    "pothole": {"department": "Public Works", "action": "Dispatch pothole repair crew"},
    "missed_trash_pickup": {"department": "Sanitation", "action": "Schedule makeup collection"},
    "illegal_dumping": {"department": "Sanitation", "action": "Dispatch cleanup crew and investigate"},
    "noise_complaint": {"department": "Code Enforcement", "action": "Send officer to assess noise violation"},
    "water_leak": {"department": "Water Utility", "action": "Dispatch emergency water crew"},
    "graffiti": {"department": "Public Works", "action": "Schedule graffiti removal"},
    "downed_tree": {"department": "Parks & Forestry", "action": "Dispatch crew to clear tree/limb"},
    "abandoned_vehicle": {"department": "Code Enforcement", "action": "Tag vehicle and initiate towing process"},
    "streetlight_outage": {"department": "Public Works", "action": "Dispatch electrician to repair streetlight"},
    "animal_control": {"department": "Animal Control", "action": "Dispatch animal control officer"},
}

STREETS = ["Maple St", "5th Ave", "Union Blvd", "Birch Rd", "Harbor Dr", "Elm St",
           "Lincoln Ave", "Cedar Ln", "Franklin St", "Riverside Dr", "Oak St", "Grant Ave"]

SEVERITY_WORDS = {
    "low": ["whenever someone gets a chance", "not urgent but", "at some point"],
    "medium": ["please take care of this soon", "this has been going on for a few days"],
    "high": ["this is dangerous", "someone could get hurt", "please send someone ASAP"],
}

MESSY_TEMPLATES = [
    "hi there theres a {issue} on {street} near the corner its been like this for {days} days {sev}",
    "URGENT!!! {issue} at {street}, {sev} can someone come out",
    "not sure who to tell but {issue} by {street}. {sev}",
    "so this is the third time im calling about {issue} on {street}... {sev} thanks",
    "{issue} on {street} sorry for typos im on my phone. {sev}",
]

ISSUE_PHRASES = {
    "pothole": "a huge pothole thats gonna wreck someones tire",
    "missed_trash_pickup": "my trash wasnt picked up again this week",
    "illegal_dumping": "someone dumped a bunch of furniture and trash",
    "noise_complaint": "the neighbors have had a party blasting music since last night",
    "water_leak": "water just gushing out of a pipe in the street",
    "graffiti": "graffiti all over the wall of the building",
    "downed_tree": "a big tree branch fell and its blocking the sidewalk",
    "abandoned_vehicle": "a car with no plates thats been sitting there for weeks",
    "streetlight_outage": "the streetlight has been out and its pitch black at night",
    "animal_control": "a stray dog thats been wandering around aggressively",
}

def make_example(rng: random.Random):
    category = rng.choice(list(CATEGORIES.keys()))
    street = rng.choice(STREETS)
    days = rng.randint(1, 14)
    priority = rng.choice(["low", "medium", "high"])
    sev_phrase = rng.choice(SEVERITY_WORDS[priority])
    template = rng.choice(MESSY_TEMPLATES)
    complaint = template.format(issue=ISSUE_PHRASES[category], street=street, days=days, sev=sev_phrase)

    meta = CATEGORIES[category]
    ticket = {
        "category": category,
        "department": meta["department"],
        "priority": priority,
        "location": street,
        "description": ISSUE_PHRASES[category][0].upper() + ISSUE_PHRASES[category][1:],
        "requested_action": meta["action"],
    }
    return {"complaint": complaint, "ticket": ticket}

def build_dataset(n, seed):
    rng = random.Random(seed)
    seen = set()
    examples = []
    while len(examples) < n:
        ex = make_example(rng)
        key = (ex["complaint"], json.dumps(ex["ticket"], sort_keys=True))
        if key in seen:
            continue
        seen.add(key)
        examples.append(ex)
    return examples

all_examples = build_dataset(360, seed=SEED)
random.Random(SEED).shuffle(all_examples)

n_test, n_val = 50, 30
test_examples = all_examples[:n_test]
val_examples = all_examples[n_test:n_test + n_val]
train_examples = all_examples[n_test + n_val:]

print(f"train={len(train_examples)} val={len(val_examples)} test(held-out)={len(test_examples)}")

for name, data in [("train", train_examples), ("val", val_examples), ("test", test_examples)]:
    with open(os.path.join(OUT_DIR, f"sft_{name}.jsonl"), "w") as f:
        for ex in data:
            f.write(json.dumps(ex) + "\n")


### Prompt format

Every model call uses the same instruction wrapper, so base/QLoRA/DPO are compared on identical prompts.

In [ ]:
SYSTEM_PROMPT = (
    "You are a municipal 311 dispatch assistant. Read the resident's complaint and output ONLY a JSON "
    "object with exactly these keys: category, department, priority, location, description, requested_action. "
    "priority must be one of: low, medium, high. Output valid JSON and nothing else."
)

def build_prompt(tokenizer, complaint: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": complaint},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def target_json(ticket: dict) -> str:
    return json.dumps(ticket, ensure_ascii=False)


## 2. Preference dataset for DPO

`chosen` = the correct ticket. `rejected` = the same ticket with one rule-based corruption applied
(dropped field, wrong category, malformed JSON, or a hallucinated location) — a realistic failure mode
for this task, not a random string.

In [ ]:
def corrupt_ticket(ticket: dict, rng: random.Random) -> str:
    corrupted = dict(ticket)
    corruption = rng.choice(["drop_field", "wrong_category", "malformed_json", "hallucinate_location"])

    if corruption == "drop_field":
        drop_key = rng.choice(["priority", "department", "requested_action"])
        del corrupted[drop_key]
        return json.dumps(corrupted, ensure_ascii=False)

    if corruption == "wrong_category":
        other = rng.choice([c for c in CATEGORIES if c != corrupted["category"]])
        corrupted["category"] = other
        corrupted["department"] = CATEGORIES[other]["department"]
        return json.dumps(corrupted, ensure_ascii=False)

    if corruption == "malformed_json":
        s = json.dumps(corrupted, ensure_ascii=False)
        return s[:-1]  # drop closing brace -> invalid JSON

    if corruption == "hallucinate_location":
        corrupted["location"] = rng.choice(STREETS) + " (unverified, not mentioned by resident)"
        return json.dumps(corrupted, ensure_ascii=False)

pref_rng = random.Random(SEED + 1)
pref_source = train_examples[: min(180, len(train_examples))]
preference_examples = []
for ex in pref_source:
    preference_examples.append({
        "prompt": ex["complaint"],
        "chosen": target_json(ex["ticket"]),
        "rejected": corrupt_ticket(ex["ticket"], pref_rng),
    })

with open(os.path.join(OUT_DIR, "dpo_pairs.jsonl"), "w") as f:
    for ex in preference_examples:
        f.write(json.dumps(ex) + "\n")

print(f"preference pairs: {len(preference_examples)}")


## 3. Load the base model and run the baseline

Loaded once in 4-bit; reused as the frozen base under every LoRA adapter later, so we never pay for a
second copy of the base weights.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

def load_base_model():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.bfloat16,
    )


In [ ]:
@torch.no_grad()
def generate_ticket(model, tokenizer, complaint: str, max_new_tokens=120) -> str:
    prompt = build_prompt(tokenizer, complaint)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()

def evaluate_model(model, tokenizer, examples, tag: str):
    """Generate on held-out complaints and score against ground truth. Every number here comes
    from the generations produced in this call -- nothing is precomputed."""
    rows = []
    for ex in examples:
        raw = generate_ticket(model, tokenizer, ex["complaint"])
        rows.append({"complaint": ex["complaint"], "gold": ex["ticket"], "raw_output": raw})

    required_keys = {"category", "department", "priority", "location", "description", "requested_action"}
    n = len(rows)
    valid_json = 0
    complete_fields = 0
    category_correct = 0

    for r in rows:
        try:
            parsed = json.loads(r["raw_output"])
            r["parsed"] = parsed
        except Exception:
            r["parsed"] = None
            continue
        valid_json += 1
        if required_keys.issubset(parsed.keys()):
            complete_fields += 1
        if parsed.get("category") == r["gold"]["category"]:
            category_correct += 1

    metrics = {
        "tag": tag,
        "n": n,
        "json_valid_rate": round(valid_json / n, 3),
        "field_complete_rate": round(complete_fields / n, 3),
        "category_accuracy": round(category_correct / n, 3),
    }
    with open(os.path.join(OUT_DIR, f"eval_{tag}.json"), "w") as f:
        json.dump({"metrics": metrics, "rows": rows}, f, indent=2)
    print(metrics)
    return metrics


In [ ]:
base_model = load_base_model()
base_model.eval()

def run_base_eval():
    return evaluate_model(base_model, tokenizer, test_examples, tag="base")

base_metrics, _ = profile_run("baseline_eval", run_base_eval)


## 4. QLoRA supervised fine-tuning

- 4-bit NF4 base (already loaded) + LoRA adapters on attention and MLP projections.
- `paged_adamw_8bit` optimizer, gradient checkpointing on, bf16 compute.
- Small per-device batch with gradient accumulation to reach a stable effective batch size without
  paying for large per-step activation memory.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

def build_sft_dataset(examples):
    input_ids_list, labels_list = [], []
    for ex in examples:
        prompt = build_prompt(tokenizer, ex["complaint"])
        target = target_json(ex["ticket"]) + tokenizer.eos_token

        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]

        input_ids = (prompt_ids + target_ids)[:MAX_SEQ_LEN]
        labels = ([-100] * len(prompt_ids) + target_ids)[:MAX_SEQ_LEN]
        input_ids_list.append(input_ids)
        labels_list.append(labels)
    return Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})

sft_train_ds = build_sft_dataset(train_examples)
sft_val_ds = build_sft_dataset(val_examples)

def pad_collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attn = [], [], []
    pad_id = tokenizer.pad_token_id
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [pad_id] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
        attn.append([1] * len(x["input_ids"]) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attn),
    }


In [ ]:
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

def make_sft_model():
    m = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
    m = get_peft_model(m, LORA_CONFIG)
    m.print_trainable_parameters()
    return m

sft_model = make_sft_model()


In [ ]:
# Safe, memory-bounded config -- this is the run we expect to succeed.
SAFE_ARGS = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "sft_ckpt"),
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size 16
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    report_to=[],
    seed=SEED,
)

def run_sft_training():
    trainer = Trainer(
        model=sft_model,
        args=SAFE_ARGS,
        train_dataset=sft_train_ds,
        eval_dataset=sft_val_ds,
        data_collator=pad_collate,
    )
    trainer.train()
    return trainer

sft_trainer, sft_train_record = profile_run("qlora_sft_safe_config", run_sft_training)
sft_model.save_pretrained(os.path.join(OUT_DIR, "qlora_adapter"))


In [ ]:
def run_sft_eval():
    sft_model.eval()
    return evaluate_model(sft_model, tokenizer, test_examples, tag="qlora")

qlora_metrics, _ = profile_run("qlora_eval", run_sft_eval)


## 5. Deliberate OOM: create it, diagnose it, fix it

This is not the safe config above. We push per-device batch size and sequence length up and turn
gradient checkpointing off, on the same 0.5B model, to genuinely exceed the T4's memory rather than
simulate it.

**Why this should OOM:** with checkpointing off, every layer's activations are kept for the backward
pass, and self-attention memory scales roughly with `batch_size x seq_len^2 x num_layers`. Even on a
tiny model, batch=32 at seq_len=1024 without checkpointing multiplies that activation footprint by
roughly two orders of magnitude versus the safe config (batch=4, seq_len<=256, checkpointing on).

If this particular config does not OOM on your Colab instance (free-tier GPU allocation varies), raise
`UNSAFE_BATCH_SIZE` and/or `UNSAFE_SEQ_LEN` further and rerun this cell until it does — the point is to
observe a genuine `torch.cuda.OutOfMemoryError`, not to hit these exact numbers.

In [ ]:
UNSAFE_BATCH_SIZE = 32
UNSAFE_SEQ_LEN = 1024

def build_padded_batch(n, seq_len):
    # Synthetic max-length batch: isolates the memory effect of batch size / seq length / checkpointing
    # from data-length variance, so the OOM (or lack of it) is attributable to the config change alone.
    input_ids = torch.randint(0, tokenizer.vocab_size, (n, seq_len))
    labels = input_ids.clone()
    attn = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attn}

def run_unsafe_step():
    unsafe_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=False)
    unsafe_model = get_peft_model(unsafe_model, LORA_CONFIG)
    unsafe_model.gradient_checkpointing_disable()
    unsafe_model.train()

    optimizer = torch.optim.AdamW(unsafe_model.parameters(), lr=2e-4)
    batch = {k: v.to(unsafe_model.device) for k, v in build_padded_batch(UNSAFE_BATCH_SIZE, UNSAFE_SEQ_LEN).items()}

    outputs = unsafe_model(**batch)
    outputs.loss.backward()
    optimizer.step()
    return outputs.loss.item()

oom_captured = False
try:
    _, oom_record = profile_run("deliberate_oom_attempt", run_unsafe_step)
except torch.cuda.OutOfMemoryError:
    oom_captured = True
    tb = traceback.format_exc()
    with open(os.path.join(OUT_DIR, "oom_traceback.txt"), "w") as f:
        f.write(tb)
    log_result({"stage": "deliberate_oom_attempt", "outcome": "CUDA_OOM", "config": {
        "batch_size": UNSAFE_BATCH_SIZE, "seq_len": UNSAFE_SEQ_LEN, "grad_checkpointing": False}})
    print("Captured a genuine CUDA OOM -- see outputs/oom_traceback.txt")
    print(tb[-1500:])
    torch.cuda.empty_cache()

print("OOM captured:", oom_captured)


### Diagnosis (fill this in from your actual `outputs/oom_traceback.txt` after running the cell above)

- **What failed:** `torch.cuda.OutOfMemoryError` raised during `<paste the exact call from your traceback: forward / backward / optimizer.step>`.
- **Root cause:** activation memory for the unsafe config scales with `batch_size x seq_len^2 x num_layers` with checkpointing off; at batch=`UNSAFE_BATCH_SIZE`/seq_len=`UNSAFE_SEQ_LEN` this exceeds the ~15GB T4 budget even though the base model itself only needs a few hundred MB in 4-bit.
- **Why the safe config avoids it:** gradient checkpointing trades recomputation for not storing intermediate activations; a small per-device batch with gradient accumulation reaches the same effective batch size at a fraction of the peak activation memory; capping `MAX_SEQ_LEN` near the real data length avoids paying for padding-driven quadratic attention cost that the data never needed.

In [ ]:
# Fix: revert to the safe, bounded config and rerun -- this should complete without OOM.
def run_fixed_step():
    fixed_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
    fixed_model = get_peft_model(fixed_model, LORA_CONFIG)
    fixed_model.train()

    optimizer = torch.optim.AdamW(fixed_model.parameters(), lr=2e-4)
    batch = {k: v.to(fixed_model.device) for k, v in build_padded_batch(
        n=SAFE_ARGS.per_device_train_batch_size, seq_len=MAX_SEQ_LEN).items()}

    outputs = fixed_model(**batch)
    outputs.loss.backward()
    optimizer.step()
    del fixed_model
    return outputs.loss.item()

_, fixed_record = profile_run("post_fix_step", run_fixed_step)
print("Fixed-config step completed. Compare peak_gpu_mem_gb against the deliberate_oom_attempt record above"
      " (if it was captured) to see the actual memory delta.")


## 6. DPO on top of the QLoRA model

Key memory decision: pass `ref_model=None` with a `PeftModel` as `model`. TRL then uses the same base
weights with adapters disabled as the reference policy, so we never load a second full copy of the base
model — only one extra (tiny) LoRA adapter's worth of parameters for the policy side.

In [ ]:
from trl import DPOConfig, DPOTrainer
from datasets import load_dataset

dpo_dataset = load_dataset("json", data_files=os.path.join(OUT_DIR, "dpo_pairs.jsonl"))["train"]

def add_full_prompt(row):
    return {"prompt": build_prompt(tokenizer, row["prompt"])}

dpo_dataset = dpo_dataset.map(add_full_prompt)

dpo_args = DPOConfig(
    output_dir=os.path.join(OUT_DIR, "dpo_ckpt"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    beta=0.1,
    max_length=MAX_SEQ_LEN + 64,
    max_prompt_length=MAX_SEQ_LEN,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    report_to=[],
    seed=SEED,
)

def run_dpo_training():
    dpo_trainer = DPOTrainer(
        model=sft_model,        # PeftModel already trained with QLoRA SFT
        ref_model=None,         # None + PeftModel -> TRL reuses the base with adapters disabled as reference
        args=dpo_args,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )
    dpo_trainer.train()
    return dpo_trainer

dpo_trainer, dpo_train_record = profile_run("dpo_training", run_dpo_training)
sft_model.save_pretrained(os.path.join(OUT_DIR, "dpo_adapter"))


In [ ]:
def run_dpo_eval():
    sft_model.eval()
    return evaluate_model(sft_model, tokenizer, test_examples, tag="dpo")

dpo_metrics, _ = profile_run("dpo_eval", run_dpo_eval)


## 7. Results

Both tables below are generated from `outputs/results.json` and the per-stage `outputs/eval_*.json`
files written by the cells above -- run the notebook top to bottom first.

In [ ]:
import pandas as pd

with open(RESULTS_PATH) as f:
    perf_records = json.load(f)
perf_df = pd.DataFrame(perf_records)
perf_df


In [ ]:
compare_rows = []
for tag in ["base", "qlora", "dpo"]:
    path = os.path.join(OUT_DIR, f"eval_{tag}.json")
    if os.path.exists(path):
        with open(path) as f:
            compare_rows.append(json.load(f)["metrics"])
compare_df = pd.DataFrame(compare_rows)
compare_df


## 8. Limitations

- All complaint/ticket data is synthetically generated by templates, not real 311 records — quality
  results describe this task's synthetic distribution, not real-world resident language.
- The preference dataset's `rejected` examples are rule-corrupted rather than sampled from the model's
  own failures; a more realistic DPO setup would mine rejects from the SFT model's actual bad
  generations.
- `results.json` and the `eval_*.json` files are the source of truth for every number reported from this
  notebook. If a section wasn't (re)run, don't quote its numbers as current.